# Разметка и сравнение train guide_v2

Обе версии train оценены на одном наборе `test_guide_v2` с одинаковым разбиением и Char‑LR при пороге 0,50. В общем результате каждое нормализованное объявление с данным исходным `mcId` учитывается один раз. Для `hard_empty` важна доля полностью пустых предсказаний.

Разметка train прошла скрининг по полным текстам. Он помог найти кандидатов в обе стороны по `106`. Для синтетики также проверили сигналы остальных категорий и связь с исходными заданиями. Статус независимой смысловой проверки каждой из 5 311 строк не подтверждён. По спорным решениям оставлены записи с цитатами.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'dataset_review' / 'compare_guide_v2.py').exists())
sys.path.insert(0, str(ROOT / 'dataset_review'))

from compare_guide_v2 import OUTPUT, TEST_OUT, TRAIN_OUT, VERSION_PATHS, compare, fingerprint

RERUN = False

if RERUN:
    result = compare()
else:
    result = json.loads(OUTPUT.read_text(encoding='utf-8'))

assert result['test_sha256'] == {
    name: fingerprint(TEST_OUT / name) for name in result['test_sha256']
}
assert all(result['runs'][version]['train_sha256'] == fingerprint(path)
           for version, path in VERSION_PATHS.items())
assert len({run['fit_item_ids_sha256'] for run in result['runs'].values()}) == 1

display(pd.DataFrame([
    {'Версия': version, 'Строк в fit': run['fit_rows'], 'Реальные': run['fit_by_origin']['real'],
     'Синтетические': run['fit_by_origin']['synthetic'],
     'Пересечения fit и теста': run['fit_eval_overlap']}
    for version, run in result['runs'].items()
]))
print('Повторных строк между тестовыми срезами:', result['duplicate_evaluation_rows'])
print('Dev-строк из stress, которые встречаются в полном train:',
      len(result['runs']['guide_v2']['train_full_stress_overlaps']))

,Версия,Строк в fit,Реальные,Синтетические,Пересечения fit и теста
0,guide_v1,4351,3404,947,0
1,guide_v2,4351,3404,947,0


Повторных строк между тестовыми срезами: 95
Dev-строк из stress, которые встречаются в полном train: 14


## Сравнение на тестовых наборах

Пропуск `106` означает, что модель не вернула эту метку при положительном эталоне. Ложный `106` означает, что модель вернула метку без подтверждения в эталоне.

In [2]:
slices = [
    'unique_all', 'gold', 'hard', 'stress_all',
    'gold_core', 'hard_empty', 'decoupled_solo_and_pairs',
    'context_traps_and_negations', 'dirty_text_and_homoglyphs',
    'non_101_source_mcid', 'mentor_borderline_switches',
]
old = result['runs']['guide_v1']['metrics']
new = result['runs']['guide_v2']['metrics']

def count_change(before, after):
    return f'{before} → {after}'

table = []
for name in slices:
    a, b = old[name], new[name]
    table.append({
        'Набор': name,
        'Строк': b['rows'],
        'F1 v1, %': round(a['micro_f1'] * 100, 2),
        'F1 v2, %': round(b['micro_f1'] * 100, 2),
        'Δ F1, п.п.': round((b['micro_f1'] - a['micro_f1']) * 100, 2),
        'Пропуски 106': count_change(a['per_class']['106']['fn'], b['per_class']['106']['fn']),
        'Ложные 106': count_change(a['per_class']['106']['fp'], b['per_class']['106']['fp']),
        'Полное совпадение, %': f"{a['exact_match']:.2%} → {b['exact_match']:.2%}",
    })
display(pd.DataFrame(table))

cross_source = new['non_101_source_mcid']['per_class']['101']
print(f"Категория 101 в non_101_source_mcid. "
      f"Найдено {cross_source['tp']}, пропущено {cross_source['fn']}. "
      "В train все исходные категории равны 101.")

,Набор,Строк,"F1 v1, %","F1 v2, %","Δ F1, п.п.",Пропуски 106,Ложные 106,"Полное совпадение, %"
0,unique_all,475,75.17,75.71,0.53,26 → 15,21 → 29,49.26% → 49.68%
1,gold,80,89.22,89.47,0.25,6 → 2,1 → 5,67.50% → 67.50%
2,hard,50,54.17,55.41,1.24,8 → 6,3 → 5,26.00% → 28.00%
3,stress_all,440,77.34,77.83,0.49,18 → 9,19 → 25,52.95% → 53.18%
4,gold_core,80,89.22,89.47,0.25,6 → 2,1 → 5,67.50% → 67.50%
5,hard_empty,40,0.00,0.00,0.00,0 → 0,8 → 7,40.00% → 40.00%
6,decoupled_solo_and_pairs,100,70.54,69.67,-0.87,0 → 0,1 → 4,46.00% → 45.00%
7,context_traps_and_negations,80,74.66,76.35,1.69,5 → 1,6 → 6,37.50% → 41.25%
8,dirty_text_and_homoglyphs,50,85.53,86.27,0.75,2 → 1,0 → 0,64.00% → 64.00%
9,non_101_source_mcid,50,93.79,93.79,0.00,0 → 0,0 → 0,82.00% → 82.00%


Категория 101 в non_101_source_mcid. Найдено 0, пропущено 7. В train все исходные категории равны 101.


## Изменения и спорные примеры

Журналы содержат исходный ID, прежние и новые наборы меток, цитату и причину. Копии gold в stress получают те же метки.

In [3]:
train_changes = pd.read_csv(TRAIN_OUT / 'label_changes.csv')
test_changes = pd.read_csv(TEST_OUT / 'label_changes.csv')
train_changes['Тип источника'] = train_changes['synth_task_id'].notna().map({True: 'синтетика', False: 'реальные'})
display(pd.crosstab(train_changes['Тип источника'], train_changes['new'])
        .rename(columns={0: 'Снято', 1: 'Добавлено'}))
columns = ['dataset', 'row_id', 'mcId', 'old_labels', 'new_labels', 'quote', 'reason']
with pd.option_context('display.max_colwidth', None):
    display(test_changes[columns])
    display(pd.concat([pd.read_csv(p / 'ambiguous.csv')[columns] for p in (TEST_OUT, TRAIN_OUT)],
                      ignore_index=True))

new,Снято,Добавлено
Тип источника,,
реальные,9,61
синтетика,0,476


,dataset,row_id,mcId,old_labels,new_labels,quote,reason
0,gold,6,106,[],[106],"Внутренняя отделка, шпатлёвка, гипсокартон.",Шпатлёвка указана самостоятельной отделочной работой.
1,gold,134,106,[102],"[102, 106]",шпатлёвка стен,Исполнитель прямо предлагает шпаклёвку поверхности стен.
2,hard,hard-006,103,"[102, 103, 104, 105, 106]",[102],"плитка, плиточник","Работа присутствует только в поисковом хвосте. В основном тексте предложены штукатурка, стяжка и монтаж перегородок."
3,hard,hard-006,104,"[102, 103, 104, 105, 106]",[102],"сантехник, электрик, электрика, сантехника","Работа присутствует только в поисковом хвосте. В основном тексте предложены штукатурка, стяжка и монтаж перегородок."
4,hard,hard-006,105,"[102, 103, 104, 105, 106]",[102],"электрик, электрика","Работа присутствует только в поисковом хвосте. В основном тексте предложены штукатурка, стяжка и монтаж перегородок."
5,hard,hard-006,106,"[102, 103, 104, 105, 106]",[102],"обои, покраска","Работа присутствует только в поисковом хвосте. В основном тексте предложены штукатурка, стяжка и монтаж перегородок."
6,stress,gold_core_005,106,[],[106],"Внутренняя отделка, шпатлёвка, гипсокартон.",Копия gold row_id=6. Шпатлёвка указана самостоятельной отделочной работой.
7,stress,gold_core_059,106,[102],"[102, 106]",шпатлёвка стен,Копия gold row_id=134. Исполнитель прямо предлагает шпаклёвку поверхности стен.
8,stress,non101_013,101,"[104, 105]","[101, 104, 105]","Санузел под ключ: укладка плитки и мозаики, полная разводка сантехники",Предложен комплексный ремонт санузла с отделкой и инженерными работами.
9,stress,non101_018,101,"[102, 104, 105, 106]","[101, 102, 104, 105, 106]","Ремонт квартиры с нуля: штукатурка, разводка электрики и сантехники","Предложен полный ремонт квартиры с черновыми, инженерными и чистовыми работами."


,dataset,row_id,mcId,old_labels,new_labels,quote,reason
0,gold,68,106,[],[],"Напишите в чат ""Шпаклевка"" и получите СКИДКУ 30% на материал","Описание даёт промокод и поисковый хвост. Исходный заголовок предлагает шпаклёвку стен, но заголовок не входит во вход классификатора."
1,gold,189,106,[],[],поклейка обоев под покраску,"Работа названа только в блоке характеристик объекта с артикулом. Неясно, входит ли она в предложение или приписка относится к примеру."
2,hard,hard-023,106,[],[],"заделка швов серпянкой, шпаклёвка","Неясно, относится шпаклёвка ко всем стенам или только к швам ГКЛ."
3,stress,gold_core_038,106,[],[],"Напишите в чат ""Шпаклевка"" и получите СКИДКУ 30% на материал","Копия gold row_id=68. Описание даёт промокод и поисковый хвост. Исходный заголовок предлагает шпаклёвку стен, но заголовок не входит во вход классификатора."
4,stress,gold_core_077,106,[],[],поклейка обоев под покраску,"Копия gold row_id=189. Работа названа только в блоке характеристик объекта с артикулом. Неясно, входит ли она в предложение или приписка относится к примеру."
5,stress,hard_empty_014,106,[],[],поклейка обоев под покраску,"Копия gold row_id=189. Работа названа только в блоке характеристик объекта с артикулом. Неясно, входит ли она в предложение или приписка относится к примеру."
6,train,2500381500962,106,[103],[103],Тип ремонта: ремонт с подготовкой стен под покраску,"Неясно, относится блок характеристик к текущему предложению или к примеру объекта."
7,train,2512662750046,106,"[102, 104, 105]","[102, 104, 105]",Стены под покраску,"В прайсе указан результат, но способ подготовки не раскрыт."
8,train,2514846751337,106,[],[],КАКИЕ ПОДГОТОВИТЕЛЬНЫЕ РАБОТЫ МОЖНО ОБСУДИТЬ,Предлагается обсудить подготовку. Возможность выполнения каждого вида работ остаётся открытой.
9,train,2498783750793,106,[],[],"Стены под обои, под покраску.","Короткое объявление называет результат, но не даёт понять, как мастер готовит стены."
